# 01 Tokenization：文本如何切成 Token

这一节从最前面开始：一句人类能读懂的文本，怎样变成模型可以继续处理的 token 序列。

暂时不讨论 token 如何变成向量。先把“文本如何被切开”这一步彻底弄清楚。

## 1. 为什么要先学习 Tokenization

神经网络不能直接把“我喜欢深度学习”这串文字送进矩阵运算。

在进入神经网络之前，文本通常先经历：

$$
\text{原始文本}
\rightarrow
\text{Tokenizer}
\rightarrow
\text{Token 序列}
$$

Tokenizer 可以先理解成一套切分和转换文本的规则，而 Tokenization 就是执行这套规则的过程。

### 示意图：Tokenizer 在整个流程中的位置

![Tokenizer 将原始文本切分为 Token 序列](assets/01_tokenization_pipeline.svg)

观察重点：这一阶段只是改变文本的切分形式，输出仍然是离散符号。

## 2. 什么是 Token

Token 是 Tokenizer 处理文本时使用的基本单位。

它可能是：

- 一个完整单词。
- 一个汉字。
- 单词或文字的一部分，也就是子词。
- 标点符号。
- 模型约定的特殊符号。

所以不能简单地认为“一个 token 一定等于一个词”。不同模型使用不同的 Tokenizer，同一句话可能得到不同的 token 序列。

## 3. 三种常见切分粒度

假设要处理英文单词 playing，可以有三种直观方案。

### 3.1 字符级

$$
\text{playing}
\rightarrow
[\text{p},\text{l},\text{a},\text{y},\text{i},\text{n},\text{g}]
$$

优点是需要的基本字符种类少，几乎不会遇到完全没见过的单词。缺点是序列变长，而且单个字符携带的语义较弱。

### 3.2 单词级

$$
\text{I like playing football}
\rightarrow
[\text{I},\text{like},\text{playing},\text{football}]
$$

优点是直观。缺点是自然语言中的单词数量非常多，新词、词形变化和拼写变化都会让词表迅速膨胀。

### 3.3 子词级

$$
\text{playing}
\rightarrow
[\text{play},\text{ing}]
$$

子词方案介于字符和完整单词之间。它既能复用常见片段，又能用多个片段组合出较少见的单词。现代语言模型通常采用这种思路。

### 示意图：字符、单词与子词粒度对比

![同一文本在字符级、单词级和子词级下的切分差异](assets/02_token_granularity.svg)

观察重点：粒度越小，词表通常越容易控制，但序列会变长；子词是在两者之间取平衡。

## 4. 为什么现代模型常使用子词

如果把每个完整单词都放入词表，会遇到两个问题：

1. 词表会非常大，占用更多参数和内存。
2. 没见过的新单词很难处理。

子词可以复用已有片段。例如 learn、learning、learned 可能共享 learn 这个片段。

它在两个目标之间做了折中：

$$
\begin{aligned}
\text{字符级} &:\ \text{词表较小，但序列较长} \\
\text{单词级} &:\ \text{序列较短，但词表很大} \\
\text{子词级} &:\ \text{在词表大小和序列长度之间折中}
\end{aligned}
$$

## 5. 中文文本也不一定一个汉字一个 Token

以“我喜欢深度学习”为例，下面几种结果都有可能出现：

$$
\begin{aligned}
&[\text{我},\text{喜欢},\text{深度学习}] \\
&[\text{我},\text{喜},\text{欢},\text{深},\text{度},\text{学},\text{习}] \\
&[\text{我},\text{喜欢},\text{深度},\text{学习}]
\end{aligned}
$$

具体结果取决于模型使用的 Tokenizer 和它的词表。不能只凭肉眼判断一段文字一定会得到多少个 token。

## 6. BPE、WordPiece 和 SentencePiece 先理解到什么程度

现在不必马上推导完整算法，先知道它们都在帮助模型确定“哪些文字片段值得作为 token”。

- BPE：从较小单位开始，反复合并语料中经常一起出现的片段。
- WordPiece：同样构造子词词表，但选择片段时使用的评价方式与 BPE 不完全相同。
- SentencePiece：直接把文本当作字符序列处理，常把空格也作为文本信息的一部分，适合多种语言。

此时最重要的不是记住算法细节，而是建立这个认识：token 的边界由 Tokenizer 的规则决定，不一定等于人类看到的词语边界。

## 7. 特殊 Token

Tokenizer 还可能在普通文本 token 之外加入特殊 token。

| 特殊 Token | 常见作用 |
|---|---|
| [CLS] | 放在序列开头，某些模型用它汇总整段序列的信息 |
| [SEP] | 分隔两个句子或标记序列结束 |
| [PAD] | 把不同长度的序列补到相同长度 |
| [UNK] | 表示无法被现有词表表示的内容 |
| [MASK] | 遮住某个 token，供特定训练任务使用 |

不是所有模型都使用完全相同的名称和规则，因此看到具体模型时要查看它所配套的 Tokenizer。

## 8. Tokenization 还没有产生语义向量

完成这一节后，数据只从一整段文本变成了 token 序列：

$$
\text{我喜欢深度学习}
\rightarrow
[\text{我},\text{喜欢},\text{深度},\text{学习}]
$$

这些 token 目前仍然是离散符号，还不能直接参与神经网络的矩阵运算。

下一步要用词表为每个 token 分配一个整数 ID。

## 9. 本节小结

1. Tokenization 是把原始文本转换成 token 序列的过程。
2. Token 不一定是完整单词，也可能是字符、子词、标点或特殊符号。
3. 子词方法在词表大小和序列长度之间取得折中。
4. 同一句话使用不同 Tokenizer，可能得到不同切分结果。
5. Tokenization 结束后得到的仍然是离散符号，还不是向量。

## 10. 自测问题

1. 为什么一个 token 不一定等于一个单词？
2. 字符级切分和单词级切分各有什么问题？
3. 为什么现代语言模型常使用子词？
4. 同一句中文一定会被不同模型切成相同的 token 吗？
5. [PAD] 和 [UNK] 分别解决什么问题？
6. Tokenization 完成后，模型是否已经得到了 token 的语义向量？